In [1]:
import numpy as np
import pandas as pd

bls_workbook = pd.ExcelFile('../data/raw/bls/occupation.xlsx')
print(bls_workbook.sheet_names)

for sheet in bls_workbook.sheet_names:
    preview = pd.read_excel(bls_workbook, sheet, header=None, nrows=5)
    print('=====', sheet, preview.shape)
    print(preview.iloc[:5, :10].to_string(), '\n')

['Index', 'Table 1.1', 'Table 1.2', 'Table 1.3', 'Table 1.4', 'Table 1.5', 'Table 1.6', 'Table 1.8', 'Table 1.9', 'Table 1.10', 'Table 1.11', 'Table 1.12']
===== Index (5, 1)
                                                               0
0                                       Occupation.xlsx contents
1               Table 1.1 Employment by major occupational group
2  Table 1.2 Occupational projections and worker characteristics
3                          Table 1.3 Fastest growing occupations
4                 Table 1.4 Occupations with the most job growth 

===== Table 1.1 (5, 7)
                                                                                                     0                                     1                 2                 3                                    4                                    5                                  6
0  Table 1.1 Employment by major occupational group, 2025 and projected 2035 (Employment in thousands)                      

In [2]:
census_to_onet = pd.read_parquet('../data/derived/census-onet-crosswalk.parquet')
job_exposure = pd.read_csv('../data/raw/anthropic/job_exposure.csv')
print(job_exposure.shape)
print(job_exposure['observed_exposure'].describe().round(3), '\n')

# census code -> 6-digit SOC (one row per pair; several O*NET codes can share a SOC)
census_to_soc = (census_to_onet.assign(soc=census_to_onet['onet'].str[:7])
                 [['census', 'soc']].drop_duplicates())

census_exposure_pairs = census_to_soc.merge(job_exposure, left_on='soc', right_on='occ_code', how='left')

# coverage: which SOC codes have no exposure score?
missing_soc = census_exposure_pairs[census_exposure_pairs['observed_exposure'].isna()]['soc'].unique()
print(len(missing_soc), 'SOC codes in the crosswalk have no exposure score:', sorted(missing_soc)[:20])

census_exposure = (census_exposure_pairs.groupby('census')['observed_exposure']
                   .mean().rename('ai_exposure').reset_index())
print(census_exposure['ai_exposure'].notna().sum(), 'of', len(census_exposure), 'census codes have an exposure score\n')

# sanity check: most and least exposed census occupations
census_titles = (pd.read_parquet('../data/derived/census-soc2010-crosswalk.parquet')
                 [['census', 'title']].drop_duplicates('census'))
check = census_exposure.merge(census_titles, on='census').dropna().sort_values('ai_exposure')
print('least exposed:\n', check.head(8).to_string(index=False), '\n')
print('most exposed:\n', check.tail(8).to_string(index=False))

census_exposure.to_parquet('../data/derived/census-ai-exposure.parquet', index=False)

(756, 3)
count    756.000
mean       0.077
std        0.133
min        0.000
25%        0.000
50%        0.000
75%        0.096
max        0.745
Name: observed_exposure, dtype: float64 

37 SOC codes in the crosswalk have no exposure score: ['11-2033', '11-9072', '13-1021', '13-1022', '13-1023', '13-1082', '13-2022', '13-2023', '21-1011', '21-1014', '25-2055', '25-2056', '25-3031', '25-9042', '25-9043', '27-2091', '29-1214', '29-2011', '29-2012', '29-2036']
503 of 517 census codes have an exposure score

least exposed:
  census  ai_exposure                                                                               title
   9740          0.0                                                   Tank car, truck, and ship loaders
   3630          0.0                                                                  Massage therapists
   3640          0.0                                                                   Dental assistants
   7160          0.0                                  

In [ ]:
nem = pd.read_excel(bls_workbook, 'Table 1.2', header=1)
print(list(nem.columns), '\n')

def find_col(fragment):
    hits = [c for c in nem.columns if fragment.lower() in str(c).lower()]
    assert len(hits) == 1, (fragment, hits)
    return hits[0]

nem = nem[nem[find_col('Occupation type')] == 'Line item'].rename(columns={
    find_col('Matrix code'): 'nem_code',
    find_col('Matrix title'): 'nem_title',
    find_col('Employment, 2025'): 'emp_2025',
    find_col('Employment, 2035'): 'emp_2035',
    find_col('Median annual wage'): 'median_wage',
    find_col('education'): 'entry_education',
    find_col('Work experience'): 'work_experience',
    find_col('on-the-job'): 'on_the_job_training',
})[['nem_code', 'nem_title', 'emp_2025', 'emp_2035', 'median_wage',
    'entry_education', 'work_experience', 'on_the_job_training']]
nem['median_wage'] = pd.to_numeric(nem['median_wage'], errors='coerce')  # BLS marks top-coded wages with symbols
print(nem.shape, 'detailed occupations\n')
print(nem['entry_education'].value_counts(), '\n')

# SOC -> BLS row: exact code first, otherwise the most specific combined code (trailing zeros = prefix)
nem['prefix'] = nem['nem_code'].str.rstrip('0').str.rstrip('-')
def match_nem(soc):
    exact = nem[nem['nem_code'] == soc]
    if len(exact):
        return exact.index[0]
    hits = nem[[soc.startswith(p) for p in nem['prefix']]]
    return hits['prefix'].str.len().idxmax() if len(hits) else None

soc_to_nem = pd.DataFrame({'soc': census_to_soc['soc'].unique()})
soc_to_nem['nem_row'] = soc_to_nem['soc'].map(match_nem)
print(soc_to_nem['nem_row'].isna().sum(), 'SOC codes with no BLS row:',
      sorted(soc_to_nem.loc[soc_to_nem['nem_row'].isna(), 'soc'])[:20], '\n')

pairs = (census_to_soc.merge(soc_to_nem, on='soc')
         .merge(nem.drop(columns='prefix'), left_on='nem_row', right_index=True, how='left')
         .merge(job_exposure[['occ_code', 'observed_exposure']], left_on='soc', right_on='occ_code', how='left')
         .drop_duplicates(['census', 'nem_row', 'soc']))

def summarise(g):
    g = g.dropna(subset=['emp_2025'])
    if g.empty:
        return pd.Series(np.nan, index=['emp_2025', 'growth_pct', 'median_wage', 'entry_education',
                                        'work_experience', 'on_the_job_training', 'ai_exposure'])
    weight = g.drop_duplicates('nem_row')
    exp = g.dropna(subset=['observed_exposure'])
    biggest = weight.loc[weight['emp_2025'].idxmax()]
    return pd.Series({
        'emp_2025': weight['emp_2025'].sum(),
        'growth_pct': 100 * (weight['emp_2035'].sum() / weight['emp_2025'].sum() - 1),
        'median_wage': np.average(weight['median_wage'].fillna(weight['median_wage'].mean()), weights=weight['emp_2025'])
                       if weight['median_wage'].notna().any() else np.nan,
        'entry_education': biggest['entry_education'],
        'work_experience': biggest['work_experience'],
        'on_the_job_training': biggest['on_the_job_training'],
        'ai_exposure': np.average(exp['observed_exposure'], weights=exp['emp_2025']) if len(exp) else np.nan,
    })

census_outlook = pairs.groupby('census').apply(summarise).reset_index()
print(census_outlook.shape)
print(census_outlook.isna().sum(), '\n')

check = census_outlook.merge(census_titles, on='census')
print('fastest growing:\n', check.nlargest(8, 'growth_pct')[['title', 'growth_pct', 'median_wage', 'entry_education']].to_string(index=False), '\n')
print('fastest shrinking:\n', check.nsmallest(8, 'growth_pct')[['title', 'growth_pct', 'median_wage', 'entry_education']].to_string(index=False))

census_outlook.to_parquet('../data/derived/census-outlook.parquet', index=False)

['2025 National Employment Matrix title', '2025 National Employment Matrix code', 'Occupation type', 'Employment, 2025', 'Employment, 2035', 'Employment distribution, percent, 2025', 'Employment distribution, percent, 2035', 'Employment change, numeric, 2025–35', 'Employment change, percent, 2025–35', 'Percent self-employed, 2025', 'Occupational openings, 2025–35 annual average', 'Median annual wage, dollars, 2025', 'Typical education needed for entry', 'Work experience in a related occupation', 'Typical on-the-job training needed to attain competency in the occupation', 'Related Occupational Outlook Handbook (OOH) content', 'Related Occupational Outlook Handbook (OOH) content.1'] 

(831, 8) detailed occupations

entry_education
High school diploma or equivalent    326
Bachelor's degree                    177
No formal educational credential     109
Doctoral or professional degree       73
Postsecondary nondegree award         51
Associate's degree                    48
Master's degree